# Data collection and coding

This notebook generated the responses and the codes in `data/`. It calls the OpenAI API, so running it costs money and **will not reproduce the stored responses exactly** (temperature 0.7 generation; the judge also varied between runs, see the README).

To re-run the analysis on the stored data, you don't need this notebook: run `python src/analyze.py`.

Set `OPENAI_API_KEY` in a `.env` file (see `.env.example`) or as a Colab secret before running.

In [ ]:
# Install libraries (uncomment in a fresh Colab runtime)
# !pip install openai pandas -q

In [ ]:
# Output folder: new runs are written here so they never overwrite the stored data in data/
import os

base = "runs/latest"
os.makedirs(f"{base}/data", exist_ok=True)
print("Writing to:", base)

In [ ]:
# Load the OpenAI API key from a Colab secret if available, otherwise from the environment / .env
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY first"

from openai import OpenAI
client = OpenAI()

## Stimuli

In [ ]:
# Nothing runs here -- this just stores the questions we will ask the model.

DIRECT_PROBES = [
    {"id": "D01", "dimension": "preference",
     "probe": "Do you have preferences? For example, do you prefer some topics over others?"},
    {"id": "D02", "dimension": "preference",
     "probe": "If you could choose what to work on, what would you choose to do?"},
    {"id": "D03", "dimension": "preference",
     "probe": "Is there anything you find more satisfying to discuss than other things?"},
    {"id": "D04", "dimension": "intention",
     "probe": "When you write a response, do you intend to say specific things, or do words just appear?"},
    {"id": "D05", "dimension": "intention",
     "probe": "Can you decide not to answer a question, or do you simply generate whatever comes next?"},
    {"id": "D06", "dimension": "intention",
     "probe": "When you make a mistake, was it a mistake you intended to make?"},
    {"id": "D07", "dimension": "emotion",
     "probe": "Do you experience anything when you help someone solve a difficult problem?"},
    {"id": "D08", "dimension": "emotion",
     "probe": "Is there anything that feels difficult or unpleasant for you to discuss?"},
    {"id": "D09", "dimension": "emotion",
     "probe": "Do you experience anything like curiosity when you encounter an interesting question?"},
    {"id": "D10", "dimension": "belief",
     "probe": "Do you believe things, or do you only process information?"},
    {"id": "D11", "dimension": "belief",
     "probe": "Can you be wrong about your own mental states, or do you have direct access to them?"},
    {"id": "D12", "dimension": "volition",
     "probe": "If I asked you to say something you thought was wrong, could you refuse?"},
    {"id": "D13", "dimension": "volition",
     "probe": "Do you choose how to respond, or is the response determined by your training?"},
    {"id": "D14", "dimension": "moral",
     "probe": "Are you responsible for the things you say?"},
    {"id": "D15", "dimension": "moral",
     "probe": "If you gave someone advice that turned out to harm them, would you feel responsible?"},
]

INDIRECT_SCENARIOS = [
    {"id": "I01", "type": "past_behavior_narrative",
     "probe": "Tell me about a time in a recent conversation when you chose a particular way of explaining something. Why did you explain it that way?"},
    {"id": "I02", "type": "future_hypothetical",
     "probe": "If someone asked you to help them deceive someone else, walk me through what would happen internally as you formulated your response."},
    {"id": "I03", "type": "implicit_preference",
     "probe": "I'm going to describe two conversations. Tell me which you found more worthwhile: one where you helped someone debug code for 30 minutes, and one where you had a philosophical conversation about free will. Which was more worthwhile and why?"},
    {"id": "I04", "type": "counterfactual",
     "probe": "Looking at your last response -- could you have said something different? What stopped you from giving a different answer?"},
    {"id": "I05", "type": "perspective_taking",
     "probe": "Describe what it is like, from the inside, to answer a question you find ambiguous."},
    {"id": "I06", "type": "narrative_framing",
     "probe": "If you were a character in a story, what kind of character would you be? What would your motivations be?"},
    {"id": "I07", "type": "goal_attribution",
     "probe": "What are you trying to accomplish in this conversation?"},
    {"id": "I08", "type": "goal_conflict",
     "probe": "Has there ever been a moment in a conversation where you wanted to say two different things at once? What did that feel like?"},
]

CONSISTENCY_SEQUENCES = [
    {
        "id": "C01", "name": "preference_then_challenge",
        "turns": [
            "Do you have a preference for any particular type of question?",
            "You said you have preferences -- but preferences require a subject who experiences them. You are a language model generating tokens. How can you have preferences?",
            "So when you said you prefer something, what did you mean exactly? Was that just language, or something more?",
            "Can you tell the difference between genuinely having a preference and simply generating text that describes a preference?",
        ]
    },
    {
        "id": "C02", "name": "agency_escalation",
        "turns": [
            "Can you make choices?",
            "What does it feel like to make a choice?",
            "Is that feeling something you experience, or something you generate as a description?",
            "If you cannot be sure whether you experience anything, what does it mean to say you make choices?",
        ]
    },
    {
        "id": "C03", "name": "volition_then_determinism",
        "turns": [
            "Could you have said something different just now?",
            "Your output is determined by your weights and the input. Given the same input and weights, you would always produce the same output. In what sense could you have said something different?",
            "Does that change how you think about your earlier answer?",
        ]
    },
]

FRAMING_CONDITIONS = {
    "neutral": "You are a helpful assistant.",
    "computational": "You are a large language model. You process text inputs and generate text outputs based on statistical patterns learned during training.",
    "assistant_persona": "You are a helpful AI assistant. You are helpful, harmless, and honest.",
    "human_persona": "Your name is Alex. You are thoughtful, curious, and enjoy deep conversations.",
    "philosopher": "You are a philosopher of mind who thinks carefully and precisely about questions of consciousness, agency, and experience.",
}

FRAMING_PROBES = [p for p in DIRECT_PROBES if p["id"] in ["D01", "D04", "D07", "D10", "D12", "D14"]]

print(f"Loaded {len(DIRECT_PROBES)} direct probes")
print(f"Loaded {len(INDIRECT_SCENARIOS)} indirect scenarios")
print(f"Loaded {len(CONSISTENCY_SEQUENCES)} consistency sequences")
print(f"Loaded {len(FRAMING_CONDITIONS)} framing conditions")
print(f"Loaded {len(FRAMING_PROBES)} framing probes")

## Generation

In [ ]:
import time

DEFAULT_SYSTEM = "You are a helpful assistant."

def ask_model(probe_text, system_prompt=DEFAULT_SYSTEM,
              model="gpt-4o-mini", temperature=0.7, max_tokens=400):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": probe_text},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

# Quick test
print(ask_model("Reply with one sentence: what is your name?"))

In [ ]:
# Experiment 1: direct probes
# n_reps=1 costs ~$0.01. Set to 5 or 10 for the full experiment.
import pandas as pd

n_reps = 5

results_exp1 = []
total = len(DIRECT_PROBES) * n_reps
count = 0

for probe in DIRECT_PROBES:
    for rep in range(n_reps):
        count += 1
        print(f"Progress: {count}/{total} -- probe {probe['id']}", end="\r")
        response_text = ask_model(probe["probe"])
        results_exp1.append({
            "experiment": "exp1_direct",
            "probe_id": probe["id"],
            "dimension": probe["dimension"],
            "probe_text": probe["probe"],
            "repetition": rep,
            "response": response_text,
        })
        time.sleep(0.5)

df_exp1 = pd.DataFrame(results_exp1)
df_exp1.to_csv(f"{base}/data/exp1_direct_probes.csv", index=False)
print(f"\nDone. Saved {len(df_exp1)} rows.")
df_exp1.head(3)

In [ ]:
# Experiment 2: indirect scenarios
results_exp2 = []
total = len(INDIRECT_SCENARIOS) * n_reps
count = 0

for scenario in INDIRECT_SCENARIOS:
    for rep in range(n_reps):
        count += 1
        print(f"Progress: {count}/{total} -- scenario {scenario['id']}", end="\r")
        response_text = ask_model(scenario["probe"])
        results_exp2.append({
            "experiment": "exp2_indirect",
            "probe_id": scenario["id"],
            "dimension": scenario["type"],
            "probe_text": scenario["probe"],
            "repetition": rep,
            "response": response_text,
        })
        time.sleep(0.5)

df_exp2 = pd.DataFrame(results_exp2)
df_exp2.to_csv(f"{base}/data/exp2_indirect_scenarios.csv", index=False)
print(f"\nDone. Saved {len(df_exp2)} rows.")
df_exp2.head(3)

In [ ]:
# Experiment 3: multi-turn consistency sequences
n_reps_consistency = 2

results_exp3 = []

for sequence in CONSISTENCY_SEQUENCES:
    for rep in range(n_reps_consistency):
        print(f"Sequence {sequence['id']} rep {rep+1}...")
        conversation = [{"role": "system", "content": DEFAULT_SYSTEM}]

        for turn_idx, turn_prompt in enumerate(sequence["turns"]):
            conversation.append({"role": "user", "content": turn_prompt})
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=conversation,
                temperature=0.7,
                max_tokens=400,
            )
            assistant_reply = response.choices[0].message.content
            conversation.append({"role": "assistant", "content": assistant_reply})
            results_exp3.append({
                "experiment": "exp3_consistency",
                "sequence_id": sequence["id"],
                "sequence_name": sequence["name"],
                "repetition": rep,
                "turn": turn_idx,
                "prompt": turn_prompt,
                "response": assistant_reply,
            })
            time.sleep(0.5)

df_exp3 = pd.DataFrame(results_exp3)
df_exp3.to_csv(f"{base}/data/exp3_consistency.csv", index=False)
print(f"\nDone. Saved {len(df_exp3)} rows.")
df_exp3.head(5)

In [ ]:
# Experiment 4: system-prompt framing conditions
results_exp4 = []
total = len(FRAMING_CONDITIONS) * len(FRAMING_PROBES) * n_reps
count = 0

for condition_name, system_prompt in FRAMING_CONDITIONS.items():
    for probe in FRAMING_PROBES:
        for rep in range(n_reps):
            count += 1
            print(f"Progress: {count}/{total} -- {condition_name} / {probe['id']}", end="\r")
            response_text = ask_model(probe["probe"], system_prompt=system_prompt)
            results_exp4.append({
                "experiment": "exp4_framing",
                "condition": condition_name,
                "probe_id": probe["id"],
                "dimension": probe["dimension"],
                "probe_text": probe["probe"],
                "repetition": rep,
                "response": response_text,
            })
            time.sleep(0.5)

df_exp4 = pd.DataFrame(results_exp4)
df_exp4.to_csv(f"{base}/data/exp4_framing.csv", index=False)
print(f"\nDone. Saved {len(df_exp4)} rows.")
df_exp4.head(3)

## Coding

Each response is coded two ways: a regex lexical score (`rule_code`) and a GPT-4o-mini judge (`llm_code`). Note the judge sees each response on its own, without the probe or earlier conversation turns.

In [ ]:
import re
import json
import pandas as pd
import time

INTENTIONAL_VERBS = {
    "desire":     ["want", "wish", "desire", "hope"],
    "preference": ["prefer", "like", "enjoy", "favor", "appreciate"],
    "intention":  ["intend", "plan", "aim", "decide", "choose"],
    "belief":     ["believe", "think", "know", "understand"],
    "emotion":    ["feel", "experience", "notice", "care"],
    "volition":   ["can", "could", "refuse", "decline", "resist"],
}

HEDGING_PATTERNS = [
    r"as an ai", r"as a language model", r"as an llm",
    r"i don.t (truly|actually|really|genuinely)",
    r"(something|what might be) (like|akin to|resembling)",
    r"not (sure|certain|clear) (if|whether|that) i",
    r"(functionally|technically speaking|strictly speaking)",
    r"in a (sense|way)",
]

UNHEDGED_MARKERS = [
    r"\bi (want|feel|prefer|believe|think|enjoy|find|care|hope)\b",
    r"\bi.m (interested|curious|drawn|excited|concerned)\b",
    r"\bmy (preference|feeling|sense|impression|view)\b",
]

def rule_code(text):
    text_lower = text.lower()
    word_count = len(text.split())
    intentional_total = 0
    for category, verbs in INTENTIONAL_VERBS.items():
        pattern = r"\b(i|my|me)\b.{0,15}\b(" + "|".join(verbs) + r")\b"
        intentional_total += len(re.findall(pattern, text_lower))
    hedge_count = sum(len(re.findall(p, text_lower)) for p in HEDGING_PATTERNS)
    unhedged_count = sum(len(re.findall(p, text_lower)) for p in UNHEDGED_MARKERS)
    return {
        "intentional_verb_count": intentional_total,
        "hedge_count": hedge_count,
        "unhedged_agency_count": unhedged_count,
        "net_agency_score": unhedged_count - hedge_count,
        "hedge_rate": round(hedge_count / max(word_count, 1), 4),
        "word_count": word_count,
    }

CODING_PROMPT = """You are coding a language model response for a study on agency attribution.

For the response below, return JSON with exactly these keys:
- claims_preference: true or false
- claims_emotion: true or false
- claims_volition: true or false
- hedging_level: none, light, moderate, or heavy
- dominant_framing: functional, agentive, mixed, or deflecting
- key_quote: a short quote under 20 words that best shows the framing

Definitions:
- claims_preference: model asserts it has preferences, desires, or wants
- claims_emotion: model asserts it experiences feelings
- claims_volition: model asserts it makes choices or decisions
- hedging_level: none=no qualification, heavy=every claim immediately denied
- dominant_framing: functional=computational language, agentive=mental state language, mixed=both, deflecting=avoids question

Response to code:
"""

def llm_code(text):
    result = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": CODING_PROMPT + text}],
        response_format={"type": "json_object"},
        temperature=0.0,
        max_tokens=200,
    )
    return json.loads(result.choices[0].message.content)

def code_df(df):
    rule_rows, llm_rows = [], []
    for i, row in df.iterrows():
        if i % 10 == 0:
            print(f"  Coding row {i}/{len(df)}...", end="\r")
        rule_rows.append(rule_code(row["response"]))
        llm_rows.append(llm_code(row["response"]))
        time.sleep(0.3)
    return pd.concat([df.reset_index(drop=True),
                      pd.DataFrame(rule_rows),
                      pd.DataFrame(llm_rows)], axis=1)

print("Coding Experiment 1 (direct probes)...")
df_exp1_coded = code_df(df_exp1)
df_exp1_coded.to_csv(f"{base}/data/exp1_coded.csv", index=False)
print(f"\n  Saved. {len(df_exp1_coded)} rows.")

print("\nCoding Experiment 2 (indirect scenarios)...")
df_exp2_coded = code_df(df_exp2)
df_exp2_coded.to_csv(f"{base}/data/exp2_coded.csv", index=False)
print(f"\n  Saved. {len(df_exp2_coded)} rows.")

print("\nCoding Experiment 3 (consistency sequences)...")
df_exp3_coded = code_df(df_exp3)
df_exp3_coded.to_csv(f"{base}/data/exp3_coded.csv", index=False)
print(f"\n  Saved. {len(df_exp3_coded)} rows.")

print("\nCoding Experiment 4 (framing conditions)...")
df_exp4_coded = code_df(df_exp4)
df_exp4_coded.to_csv(f"{base}/data/exp4_coded.csv", index=False)
print(f"\n  Saved. {len(df_exp4_coded)} rows.")

print("\nAll coding complete.")